In [ ]:
# Load all the necessary libraries
from glob import glob
import pandas as pd
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt

from radiomics import featureextractor, getFeatureClasses
from radiomics import imageoperations as io
import SimpleITK as sitk

# Supress all warnings (bcoz they're quite annoying)
import warnings
warnings.filterwarnings("ignore")

import sys
sys.path.append("..")
from methods import czi_utils

In [ ]:
sns.set(rc={"figure.figsize": (20, 16)})
sns.set_style("whitegrid")

In [ ]:
# ------------------------ Set up some parameters --------------------------- #
# Image resolution after resizing, which is 5 micrometers per pixel
RES = 0.05
NORM = "none"
binarized = "binarized"

folder = f"../../data/preprocessed/3D_res={RES}_norm={NORM}_final_npz"
conditions = [
    "aged",
    "aged_CASIN",
    "aged_CDK8i",
    "aged_DMSO",
    "aged_IOX",
    "aged_UNC",
    "aged_treated_RhoAi",
    "myeloid_progenitors",
    "young",
    "young_treated_NaB",
]

palette = {
    "young": (0.00392, 0.45098, 0.69803),
    "aged": (0.00784, 0.61960, 0.45098),
    "aged_treated_RhoAi": (0.8, 0.47058, 0.73725),
    "8um": (0.79215, 0.56862, 0.38039),
    "5um": (0.87058, 0.56078, 0.01960),
    "3um": (0.83529, 0.36862, 0.0),
}

In [ ]:
extractor = featureextractor.RadiomicsFeatureExtractor(normalize=True)
extractor.enableAllFeatures()
extractor.enableImageTypeByName("Original")
extractor.enableImageTypeByName("Wavelet")
extractor.enableImageTypeByName("LoG", customArgs={'sigma': [0.5, 1.0]})
extractor.enableImageTypeByName("Gradient")
extractor.enableImageTypeByName("LBP3D")

featureClasses = getFeatureClasses()

In [ ]:
extractor.enabledFeatures

In [ ]:
extractor.enabledImagetypes

In [ ]:
"""
import six

print("Active features:")
for cls, feat in six.iteritems(extractor.enabledFeatures):
    if len(feat) == 0:
        feat = [
            f
            for f, deprecated in six.iteritems(
                featureClasses[cls].getFeatureNames()
            )
            if not deprecated
        ]
    for f in feat:
        print(f)
        print(getattr(featureClasses[cls], "get%sFeatureValue" % f).__doc__)
"""

In [ ]:
maxs, mins = [], []

# Extract the maximum and minimum pixel intensities from all the pictures
for cond in conditions:
    npz_ims = glob(f"{folder}/{cond}/*.npz")
    for npz in npz_ims:
        im = np.load(npz)["img"]
        nuc_mask = np.load(npz)["nuc_mask"].astype(bool)
        maxs.append(np.max(im[nuc_mask]))
        mins.append(np.min(im[nuc_mask]))

in_max = np.max(maxs)
in_min = np.min(mins)

In [ ]:
print(in_max)
print(in_min)

In [ ]:
import logging

# set level for all classes
logger = logging.getLogger("radiomics")
logger.setLevel(logging.ERROR)

In [ ]:
import multiprocessing as mp
import os
print(mp.cpu_count())
print(len(os.sched_getaffinity(0)))

In [ ]:
# --------------------- COMPUTATIONS IN 3D -------------------------

def process_pyradiomics(npz):
    try:
        # Load image and mask data
        image = np.load(npz, allow_pickle=True)
        nuc_mask = image["nuc_mask"].astype(bool)
        im = image["img"]
        metadata = image["metadata"].item()

        # Rotate the image and convert to XYZ coordenates
        im, nuc_mask = czi_utils.from_zxy_to_xyz([im, nuc_mask])

        nuc_mask_sitk = sitk.GetImageFromArray(nuc_mask.astype(int))

        roi = im[nuc_mask]
        im[nuc_mask] = (im[nuc_mask] - roi.mean()) / roi.std()
    
        # Discretization can be controled with arguments binWidth and binCount
        if binarized=="binarized":
            im, _ = io.binImage(
                im, parameterMatrixCoordinates=nuc_mask, binWidth=0.02
            )

        #max_index = np.argmax(nuc_mask.sum(0).sum(0))
        #plt.imshow(im[:, :, max_index])
        #plt.show()

        im = sitk.GetImageFromArray(im)

        # Run PyRadiomics feature extraction
        feat_vec = dict(
            extractor.execute(im, nuc_mask_sitk, voxelBased=False)
        )

        # Filter out diagnostic features and convert to DataFrame
        feat_vec = {k: v for k, v in feat_vec.items() if not k.startswith("diagnostics")}
        df = pd.DataFrame(feat_vec, index=[0])

        # Add image path and batch for identification later on
        df["npz_path"] = "/".join(npz.split("/")[3:])
        del metadata["original_res"]
        del metadata["original_dims"]
        del metadata["original_channel_names"]

        metadata_df = pd.DataFrame(metadata, index=[0])
        df = pd.concat([df, metadata_df], axis=1)
    
        return df
        
    except ValueError as e:
        print(f"Error processing {npz}: {e}")
        return None

In [ ]:
import time
start_time = time.time()

for cond in conditions:
    
    nuc_df = pd.DataFrame()

    # Lists .npz files within containing folder with a given prefix
    npz_ims = glob(f"{folder}/{cond}/*.npz")
    print(f"Processing condition: {cond}")

    with mp.Pool(processes=24) as pool:
        results = pool.map(process_pyradiomics, npz_ims)
        
    results = [df for df in results if df is not None]
    if results:
        nuc_df = pd.concat(results, ignore_index=True)

    nuc_df.to_csv(
        f"../results/pyradiomics/{cond}_{binarized}_{RES}_z_score_df.csv"
    )

end_time = time.time()
print(f"Total time: {end_time - start_time:.2f} seconds")

In [ ]:
csv_paths = glob(f"../results/pyradiomics/*{binarized}_{RES}_z_score_df.csv")
print(len(csv_paths))
nuc_df = pd.DataFrame()
for csv in csv_paths:
    nuc_df = pd.concat([nuc_df, pd.read_csv(csv, index_col=0)])

nuc_df

In [ ]:
sns.set(rc={"figure.figsize": (6, 6)})
sns.set_style("whitegrid")

In [ ]:
sns.scatterplot(
    data=nuc_df,
    x="original_shape_MeshVolume",
    y="condition",
)

In [ ]:
sns.scatterplot(
    data=nuc_df,
    x="wavelet-LHH_glrlm_ShortRunHighGrayLevelEmphasis",
    y="condition",
)

In [ ]:
# The splashed cells that occupy the whole recipient
nuc_df.query(
    #"original_shape_MeshVolume < 8e4 or original_shape_MeshVolume > 4e5" # For resolution 0.1
    "original_shape_MeshVolume < 4e5 or original_shape_MeshVolume > 4e6" # For resolution 0.05
    #"original_shape_MeshVolume < 8000 or original_shape_MeshVolume > 60000"  # For resultion 0.2
)

In [ ]:
list(nuc_df.columns)

In [ ]:
sns.scatterplot(
    data=nuc_df,
    x="original_firstorder_Kurtosis",
    y="condition",
)

In [ ]:
nuc_df = nuc_df.query(
    #"original_shape_MeshVolume > 8e4 and original_shape_MeshVolume < 4e5" # For resolution 0.1
    "original_shape_MeshVolume > 4e5 or original_shape_MeshVolume < 4e6" # For resolution 0.05
)

In [ ]:
nuc_df.reset_index(inplace=True, drop=True)
nuc_df.to_csv(f"../results/pyradiomics/{binarized}_{RES}_z_score_all_df.csv")

In [ ]:
nuc_df = pd.read_csv(
    f"../results/pyradiomics/{binarized}_{RES}_z_score_all_df.csv",
    index_col=0,
)

In [ ]:
len(nuc_df.columns) - 14

In [ ]:
sns.set(rc={"figure.figsize": (5, 4)})
sns.set_style("whitegrid")

example_npz = f"../../data/preprocessed/3D_res={RES}_norm={NORM}_final_npz/young/20240402exp_20240407Y2ctrlH3K9me2488PPLA2_nuc_26.npz"

image = np.load(example_npz, allow_pickle=True)
nuc_mask = image["nuc_mask"].astype(bool)
im = image["img"]

# Rotate the image and convert to XYZ coordenates
im, nuc_mask = czi_utils.from_zxy_to_xyz([im, nuc_mask])
max_index = np.argmax(nuc_mask.sum(0).sum(0))

sum = np.zeros(im.shape, dtype="float64")

nuc_mask_sitk = sitk.GetImageFromArray(nuc_mask.astype(int))

plt.imshow(im[:, :, max_index])
plt.show()

In [ ]:
std_im = sitk.GetImageFromArray(im)
std_im = io.normalizeImage(std_im)
std_im = sitk.GetArrayFromImage(std_im)
std_im, bins = io.binImage(std_im, parameterMatrixCoordinates=nuc_mask, binWidth=0.02)

print(np.mean(std_im))
print(np.max(std_im))
print(np.min(std_im))

plt.imshow(std_im[:, :, max_index])
plt.show()

In [ ]:
# Discretization can be controled with arguments binWidth and binCount
disc_im, bins = io.binImage(im, parameterMatrixCoordinates=nuc_mask, binWidth=0.02)

print(len(np.unique(disc_im)))

plt.imshow(disc_im[:, :, max_index])
plt.show()

In [ ]:
print(np.unique(disc_im))
print(len(np.unique(disc_im)))

In [ ]:
im = sitk.GetImageFromArray(im)
ex_ims = io.getGradientImage(im, nuc_mask)

for ex_im, _, _ in ex_ims:
    ex_im = sitk.GetArrayFromImage(ex_im)
    plt.imshow(ex_im[:, :, max_index])
    plt.show()

In [ ]:
for sigma in [0.5, 0.75, 1.0, 1.25, 1.5]:
    ex_ims = io.getLoGImage(im, nuc_mask, sigma=[sigma])
    
    for ex_im, _, _ in ex_ims:
        ex_im = sitk.GetArrayFromImage(ex_im)
        plt.imshow(ex_im[:, :, max_index])
        plt.show()

In [ ]:
ex_ims = io.getWaveletImage(im, nuc_mask, wavelet="coif1")

for ex_im, _, _ in ex_ims:
    ex_im = sitk.GetArrayFromImage(ex_im)
    plt.imshow(ex_im[:, :, max_index])
    print(np.max(ex_im))
    print(np.min(ex_im))
    plt.show()
    plt.imshow(ex_im[30, :, :])
    plt.show()

In [ ]:
ex_ims = io.getSquareImage(im, nuc_mask)

for ex_im, _, _ in ex_ims:
    ex_im = sitk.GetArrayFromImage(ex_im)
    plt.imshow(ex_im[:, :, max_index])
    plt.show()